In [ ]:
param(
    [string]$DownloadsPath = "$env:USERPROFILE\Downloads",
    [int]$DaysOld = 30,
    [string]$LogPrefix = "Cleanup_"
)

$timestamp = Get-Date -Format "yyyyMMdd_HHmmss"
$logFile = Join-Path -Path $DownloadsPath -ChildPath "$LogPrefix$timestamp.log"

function Write-LogResult {
    param(
        [string]$Status,
        [string]$Message = ""
    )
    $logLine = "$(Get-Date -Format 'yyyy-MM-dd HH:mm:ss') - $Status"
    if ($Message) {
        $logLine += ": $Message"
    }

    if (Test-Path -Path $DownloadsPath -PathType Container) {
        $logLine | Out-File -FilePath $logFile -Encoding utf8
    } else {

        Write-Host "Log cannot be written: $logFile"
        Write-Host $logLine
    }
}


try {

    if (-not (Test-Path -Path $DownloadsPath -PathType Container)) {
        Write-LogResult "FAILURE" "Downloads folder not found or inaccessible: $DownloadsPath"
        exit 1
    }


    $cutoff = (Get-Date).AddDays(-$DaysOld)

    $files = Get-ChildItem -Path $DownloadsPath -File | Where-Object {
        (-not $_.Attributes.HasFlag([System.IO.FileAttributes]::Hidden)) -and
        (-not $_.Attributes.HasFlag([System.IO.FileAttributes]::System)) -and
        ($_.Name -notlike "$LogPrefix*.log") -and
        ($_.LastWriteTime -lt $cutoff)
    }

    if ($files.Count -eq 0) {
        Write-LogResult "SUCCESS" "No files older than $DaysOld days found."
        exit 0
    }

    $shell = New-Object -ComObject Shell.Application
    $recycleBin = $shell.Namespace(0)

    foreach ($file in $files) {
        try {
            $item = $recycleBin.ParseName($file.FullName)
            if ($item) {
                $item.InvokeVerb("delete")
            }
        }
        catch {
            # Non-fatal error: skip this file and continue
            # (Per specification, no per-file logging is performed)
        }
    }

    Write-LogResult "SUCCESS"
    exit 0
}
catch {
    Write-LogResult "FAILURE" "Unexpected error: $_"
    exit 1
}